# Tong Yun San Gong: corpus and mode analysis

This notebook reproduces the numerical analysis for *Revisiting Tong Yun San Gong
Theory: Corpus Evidence for the Distinctions among Three Chinese Heptatonic Diao*
by Shuxin Meng, Fabian C. Moss, and Martin A. Rohrmeier.

The inputs are 2,240 processed Chinese Essen pieces and 219 manually annotated
heptatonic pieces. Run the cells in order to calculate and inspect the results.

The Bayesian mode model is adapted from
[Harasim, Moss, Ramirez, and Rohrmeier (2021)](https://doi.org/10.1057/s41599-020-00678-6)
and their [HistoryModes implementation](https://github.com/DCMLab/HistoryModes_DataCode).

## 1. Inputs

`data/corpus.csv` stores duration proportions after moving each score's declared
tonic to C. `data/heptatonic.csv` stores the manually examined profiles in a common
diatonic reference, C–D–E–F–G–A–B, together with *diao*, *gong*, and *diao-tou*
annotations. Filenames link the two tables. See [the data dictionary](data/README.md)
for source references, units, and field definitions.

In [ ]:
import math
from itertools import permutations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from numba import float64, int64, njit
from numba.experimental import jitclass
from sklearn.metrics import precision_recall_fscore_support, silhouette_score

DATA = Path("data")

PITCH_COLUMNS = [f"pc_{pitch}" for pitch in range(12)]
DIAO = ["xiazhi", "zhengsheng", "qingshang"]
FINAL_TONES = ["gong", "shang", "jue", "zhi", "yu"]
SEED = 42
EPOCHS = 1000
WARMUP = 100

corpus = pd.read_csv(
    DATA / "corpus.csv", keep_default_na=False, float_precision="round_trip"
)
heptatonic = pd.read_csv(
    DATA / "heptatonic.csv", keep_default_na=False, float_precision="round_trip"
)
annotated = heptatonic[["name", "diao"]].merge(
    corpus, on="name", how="left", validate="one_to_one", sort=False
)

display(heptatonic.groupby("diao", sort=False).size().rename("pieces").to_frame())

## 2. Corpus summaries

The geographical, genre, ethnic-group, and source summaries describe the corpus
and its annotated subset. The pitch-class means for Figure 4 use the
declared-tonic reference from `corpus.csv` for both groups.

In [ ]:
descriptive_rows = []
for subset, table in [("corpus", corpus), ("heptatonic", annotated)]:
    fields = ["genre_broad", "ethnic_group", "province", "source_family"]
    if subset == "heptatonic":
        fields = ["diao"] + fields
    for field in fields:
        counts = table[field].replace("", "[missing]").value_counts(sort=False)
        for value, count in counts.items():
            descriptive_rows.append({
                "subset": subset, "field": field, "value": value,
                "count": int(count), "denominator": len(table),
                "percent": 100 * count / len(table),
            })
descriptive_counts = pd.DataFrame(descriptive_rows)

pitch_mean_rows = []
for subset, table in [("corpus", corpus), ("heptatonic", annotated)]:
    means = table[PITCH_COLUMNS].mean()
    for pitch, value in enumerate(means):
        pitch_mean_rows.append({
            "subset": subset, "pitch_class": pitch,
            "mean_relative_duration": value,
        })
pitch_class_means = pd.DataFrame(pitch_mean_rows)

sample_summary = pd.Series({
    "corpus pieces": len(corpus),
    "annotated pieces": len(annotated),
    "mappable annotated pieces": (annotated["province_status"] == "resolved").sum(),
    "annotated pieces in Shanxi": (annotated["province"] == "Shanxi").sum(),
    "annotated pieces with genre": (annotated["genre"] != "").sum(),
    "annotated pieces with ethnic label": (annotated["ethnic_group"] != "").sum(),
}, name="pieces")
display(sample_summary.to_frame())

groups = {
    "shanxi": annotated["province"] == "Shanxi",
    "other_provinces": annotated["province"] != "Shanxi",
    "xiaodiao": annotated["genre_broad"] == "Xiaodiao",
    "other_named_genres": (
        (annotated["genre_broad"] != "") & (annotated["genre_broad"] != "Xiaodiao")
    ),
    "han": annotated["ethnic_group"] == "Han",
}
composition_rows = []
for group, selected in groups.items():
    pieces = annotated.loc[selected]
    count = int((pieces["diao"] == "xiazhi").sum())
    composition_rows.append({
        "group": group, "xiazhi_count": count, "piece_count": len(pieces),
        "reported_percent": round(100 * count / len(pieces), 1),
    })
sample_composition = pd.DataFrame(composition_rows)
display(sample_composition)

compact_sigla = ["WI", "WII", "I", "III", "IV", "II", "MM"]
named_publications = [
    "Zhongguo Minge Xuan", "Hanzu minge gailun", "Zhongguo Minge Jing Xuan",
    "Haozi article (Essen 1991)", "Xiaodiao article (Essen 1991)",
]
source_summary = pd.DataFrame({
    "group": ["compact_sigla", "named_publications", "own_recording"],
    "count": [
        corpus["source_family"].isin(compact_sigla).sum(),
        corpus["source_family"].isin(named_publications).sum(),
        corpus["own_recording"].sum(),
    ],
})
source_summary["denominator"] = len(corpus)
source_summary["percent"] = (100 * source_summary["count"] / len(corpus)).round(1)
display(source_summary)

display(pitch_class_means.pivot(
    index="pitch_class", columns="subset", values="mean_relative_duration"
).round(6))

Figure 5 summarizes annotated profiles in two pitch references: the common
diatonic reference and the reference with the annotated *gong* at pitch class 0.
Table 1 gives final-tone counts and percentages within each *diao*.

In [ ]:
annotated_profile_rows = []
for diao in DIAO:
    group = heptatonic.loc[heptatonic["diao"] == diao]
    common_profiles = group[PITCH_COLUMNS].to_numpy()
    gong_profiles = np.array([
        np.roll(profile, -gong)
        for profile, gong in zip(common_profiles, group["gong_pc"])
    ])
    for reference, profiles in [
        ("common_diatonic", common_profiles), ("gong", gong_profiles)
    ]:
        for pitch, value in enumerate(profiles.mean(axis=0)):
            annotated_profile_rows.append({
                "diao": diao, "reference": reference, "pitch_class": pitch,
                "mean_relative_duration": value,
            })
annotated_profiles = pd.DataFrame(annotated_profile_rows)

final_tone_rows = []
for diao in DIAO + ["all"]:
    group = heptatonic if diao == "all" else heptatonic.loc[heptatonic["diao"] == diao]
    counts = group["diao_tou"].value_counts().reindex(FINAL_TONES, fill_value=0)
    for tone, count in counts.items():
        final_tone_rows.append({
            "diao": diao, "diao_tou": tone,
            "count": int(count), "percent": 100 * count / len(group),
        })
table1_diao_tou = pd.DataFrame(final_tone_rows)
display(table1_diao_tou.pivot(index="diao", columns="diao_tou", values="percent")
        .reindex(index=DIAO + ["all"], columns=FINAL_TONES).round(3))

## 3. Model datasets

The original condition uses the 219 annotated profiles, each given a random
cyclic transposition. The bootstrap condition samples 200 pieces with replacement
within each *diao*. The Dirichlet condition estimates a distribution within each
*diao* by moments and draws 200 synthetic profiles per category.

For normalized profiles $x_i$, let $m$ be their mean and
$M_2 = \operatorname{mean}_i\sum_j x_{ij}^2$. The Dirichlet concentration is
$\alpha_0 = (1-M_2)/(M_2-\sum_j m_j^2)$, with
$\alpha_j=\max(\alpha_0m_j,10^{-12})$.
The class labels are used for resampling and evaluation; model inference uses
the twelve pitch-class values.

In [ ]:
def estimate_dirichlet(profiles):
    normalized = profiles.astype(float) / profiles.sum(axis=1, keepdims=True)
    mean_profile = normalized.mean(axis=0)
    second_moment = float(np.mean(np.sum(normalized ** 2, axis=1)))
    concentration = (
        (1.0 - second_moment)
        / (second_moment - float(np.sum(mean_profile ** 2)))
    )
    return np.clip(concentration * mean_profile, 1e-12, None)


by_diao = {
    diao: heptatonic.loc[heptatonic["diao"] == diao, PITCH_COLUMNS].to_numpy()
    for diao in DIAO
}
names_by_diao = {
    diao: heptatonic.loc[heptatonic["diao"] == diao, "name"].to_numpy()
    for diao in DIAO
}

bootstrap_blocks = []
bootstrap_names = []
dirichlet_blocks = []
dirichlet_parameters = []
for index, diao in enumerate(DIAO):
    profiles = by_diao[diao]
    rng = np.random.default_rng(SEED + index)
    selected = rng.integers(0, len(profiles), size=200)
    bootstrap_blocks.append(profiles[selected])
    bootstrap_names.extend(names_by_diao[diao][selected])

    alpha = estimate_dirichlet(profiles)
    dirichlet_parameters.append({"diao": diao, **dict(zip(PITCH_COLUMNS, alpha))})
    rng = np.random.default_rng(SEED + 10 + index)
    dirichlet_blocks.append(rng.dirichlet(alpha, size=200))

bootstrap = pd.DataFrame(np.vstack(bootstrap_blocks), columns=PITCH_COLUMNS)
bootstrap["diao"] = np.repeat(DIAO, 200)
bootstrap["name"] = bootstrap_names
bootstrap = bootstrap.sample(frac=1, random_state=SEED).reset_index(drop=True)

dirichlet = pd.DataFrame(np.vstack(dirichlet_blocks), columns=PITCH_COLUMNS)
dirichlet["diao"] = np.repeat(DIAO, 200)
dirichlet["name"] = ""
dirichlet = dirichlet.sample(frac=1, random_state=SEED).reset_index(drop=True)

rng = np.random.default_rng(SEED)
original = heptatonic.copy()
original[PITCH_COLUMNS] = heptatonic[PITCH_COLUMNS].apply(
    lambda profile: np.roll(profile.to_numpy(), rng.integers(0, 12)),
    axis=1, result_type="expand",
)
original["sample_id"] = original["name"]
for table in (bootstrap, dirichlet):
    table["sample_id"] = [f"sample{index:03d}" for index in range(1, len(table) + 1)]

datasets = {"original": original, "bootstrap": bootstrap, "dirichlet": dirichlet}

dirichlet_parameters = pd.DataFrame(dirichlet_parameters)

## 4. Bayesian mode model

Each piece has a latent root and mode. The model converts each duration profile
to counts with `floor(profile * 60)`. Dirichlet prior counts are 2 per root,
2 per mode, and 1 per pitch class within each mode.

A collapsed Gibbs sampler updates each piece's joint root/mode assignment.
There are 1,000 sweeps, with the first 100 discarded for prediction. A piece's
prediction is its most frequent joint assignment after warmup. Inferred mode
profiles and prevalence are calculated from the final state, including prior
counts.

The functions below implement the model of Harasim et al. (2021). Numba compiles
the sampling calculations. Distances are the minimum squared Euclidean
distance over all twelve cyclic transpositions.

In [ ]:
@njit
def seed_sampler(seed):
    """Seed the random stream used by the compiled Gibbs sampler."""
    np.random.seed(seed)


def set_seed(seed):
    """Seed NumPy and the compiled sampler before model initialization."""
    np.random.seed(seed)
    seed_sampler(seed)


@njit
def log_beta(a, b):
    return math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)


@njit
def unique_counts(values):
    sorted_values = np.sort(values.flatten())
    unique = list(sorted_values[:1])
    counts = [1.0 for _ in unique]
    for value in sorted_values[1:]:
        if value != unique[-1]:
            unique.append(value)
            counts.append(1.0)
        else:
            counts[-1] += 1.0
    return np.array(unique), np.array(counts)


@njit
def categorical_draw(probabilities):
    draw = np.random.random()
    cumulative = 0.0
    for index in range(probabilities.size):
        cumulative += probabilities[index]
        if draw < cumulative:
            return index
    return probabilities.size - 1


@njit
def matrix_choice(probabilities):
    n_rows, n_columns = probabilities.shape
    index = categorical_draw(probabilities.ravel())
    return index // n_columns, index % n_columns


@njit
def mode_distance(first_profile, second_profile):
    """Minimum squared Euclidean distance over pitch-class rotations."""
    minimum = np.inf
    for shift in range(len(first_profile)):
        distance = np.sum((np.roll(first_profile, shift) - second_profile) ** 2)
        if distance < minimum:
            minimum = distance
    return minimum


@njit
def mode_distance_matrix(profiles):
    n_pieces = len(profiles)
    distances = np.zeros((n_pieces, n_pieces))
    for first in range(n_pieces):
        for second in range(first + 1, n_pieces):
            distance = mode_distance(profiles[first], profiles[second])
            distances[first, second] = distance
            distances[second, first] = distance
    return distances

In [ ]:
STATE_FIELDS = [
    ("pitch_counts", float64[:, :]),
    ("n_pieces", int64),
    ("n_pitch_classes", int64),
    ("n_modes", int64),
    ("roots", int64[:]),
    ("modes", int64[:]),
    ("root_counts", float64[:]),
    ("mode_counts", float64[:]),
    ("mode_pitch_counts", float64[:, :]),
]


@jitclass(STATE_FIELDS)
class GibbsState:
    """Assignments and Dirichlet counts, including the prior counts."""

    def __init__(self, pitch_counts, root_prior, mode_prior, pitch_prior):
        self.pitch_counts = pitch_counts
        self.n_pieces, self.n_pitch_classes = pitch_counts.shape
        self.n_modes = mode_prior.shape[0]

        self.roots = np.random.randint(0, self.n_pitch_classes, size=self.n_pieces)
        self.modes = np.random.randint(0, self.n_modes, size=self.n_pieces)

        self.root_counts = root_prior.copy()
        values, counts = unique_counts(self.roots)
        self.root_counts[values] += counts

        self.mode_counts = mode_prior.copy()
        values, counts = unique_counts(self.modes)
        self.mode_counts[values] += counts

        self.mode_pitch_counts = pitch_prior.copy()
        for counts, root, mode in zip(self.pitch_counts, self.roots, self.modes):
            self.mode_pitch_counts[mode] += np.roll(counts, -root)

    def log_categorical_predictive(self, index, prior_counts):
        total = np.sum(prior_counts)
        return log_beta(total, 1.0) - log_beta(prior_counts[index], 1.0)

    def log_multinomial_predictive(self, counts, prior_counts):
        n_notes = float(np.sum(counts))
        total = float(np.sum(prior_counts))
        nonzero = np.nonzero(counts)
        terms = 0.0
        for count, prior_count in zip(counts[nonzero], prior_counts[nonzero]):
            terms += math.log(count) + log_beta(prior_count, count)
        return math.log(n_notes) + log_beta(total, n_notes) - terms

    def full_resample(self):
        for piece_index in range(self.n_pieces):
            self.single_resample(piece_index)

    def single_resample(self, piece_index):
        pitch_counts = self.pitch_counts[piece_index]

        # Remove this piece before evaluating its joint root/mode conditional.
        root, mode = self.roots[piece_index], self.modes[piece_index]
        self.root_counts[root] -= 1.0
        self.mode_counts[mode] -= 1.0
        self.mode_pitch_counts[mode] -= np.roll(pitch_counts, -root)

        log_joint = np.zeros((self.n_pitch_classes, self.n_modes))
        for root_candidate in range(self.n_pitch_classes):
            for mode_candidate in range(self.n_modes):
                root_logprob = self.log_categorical_predictive(
                    root_candidate, self.root_counts
                )
                mode_logprob = self.log_categorical_predictive(
                    mode_candidate, self.mode_counts
                )
                profile_logprob = self.log_multinomial_predictive(
                    pitch_counts,
                    np.roll(self.mode_pitch_counts[mode_candidate], root_candidate),
                )
                log_joint[root_candidate, mode_candidate] = (
                    root_logprob + mode_logprob + profile_logprob
                )

        log_normalizer = -np.inf
        for log_probability in log_joint.ravel():
            log_normalizer = np.logaddexp(log_normalizer, log_probability)
        joint = np.exp(log_joint - log_normalizer)

        root, mode = matrix_choice(joint)
        self.roots[piece_index] = root
        self.modes[piece_index] = mode
        self.root_counts[root] += 1.0
        self.mode_counts[mode] += 1.0
        self.mode_pitch_counts[mode] += np.roll(pitch_counts, -root)

In [ ]:
def fit_mode_model(profiles, n_modes, seed=42, epochs=1000, warmup=100):
    """Fit from duration-weighted relative pitch-class frequencies.

    The reported procedure multiplies profiles by 60 and floors each bin.
    Predictions use each piece's most frequent joint (root, mode) assignment
    after warmup. Mode templates and prevalence use the final Gibbs state.
    Pass seed=None to continue an already seeded sequence of model fits.
    """
    if seed is not None:
        set_seed(seed)

    n_pieces, n_pitch_classes = profiles.shape
    pitch_counts = np.floor(profiles * 60).astype(float)
    root_prior = 2.0 * np.ones(n_pitch_classes)
    mode_prior = 2.0 * np.ones(n_modes)
    pitch_prior = np.ones((n_modes, n_pitch_classes))
    state = GibbsState(pitch_counts, root_prior, mode_prior, pitch_prior)

    root_chain = -np.ones((epochs, n_pieces), dtype=int)
    mode_chain = -np.ones((epochs, n_pieces), dtype=int)
    for epoch in range(epochs):
        state.full_resample()
        root_chain[epoch, :] = state.roots
        mode_chain[epoch, :] = state.modes

    joint_samples = np.stack(
        (root_chain[warmup:, :], mode_chain[warmup:, :]), axis=2
    )
    root_predictions = -np.ones(n_pieces, dtype=int)
    mode_predictions = -np.ones(n_pieces, dtype=int)
    for piece_index in range(n_pieces):
        assignments, counts = np.unique(
            joint_samples[:, piece_index, :], axis=0, return_counts=True
        )
        root, mode = assignments[counts.argmax()]
        root_predictions[piece_index] = root
        mode_predictions[piece_index] = mode

    return {
        "root_predictions": root_predictions,
        "mode_predictions": mode_predictions,
        "root_counts": state.root_counts.copy(),
        "mode_counts": state.mode_counts.copy(),
        "mode_pitch_counts": state.mode_pitch_counts.copy(),
        "final_roots": state.roots.copy(),
        "final_modes": state.modes.copy(),
        "root_chain": root_chain,
        "mode_chain": mode_chain,
        "pitch_counts": pitch_counts,
    }

## 5. Evaluation

For the three-mode solution, modes are numbered 0–2 in decreasing order of
final-state prevalence. A one-to-one mapping to *diao* labels maximizes agreement
with the annotations and determines precision, recall, and F1. Confusion counts
retain the numbered modes.

Mode profiles are aligned for inspection using consecutive whole-tone steps
to locate *gong*. The profile table records the alignment pitch class.
Silhouette scores use the transposition-invariant distance matrix and the
predicted mode of each piece.

In [ ]:
def align_mode_profile(profile):
    filtered = np.where(profile < 0.01, 0, profile)
    ranked = np.argsort(-filtered)[:np.count_nonzero(filtered)]
    pitches = sorted(ranked)
    extended = pitches + [pitch + 12 for pitch in pitches[:2]]
    triples = [
        [pitch % 12 for pitch in extended[index:index + 3]]
        for index in range(len(extended) - 2)
        if np.array_equal(np.diff(extended[index:index + 3]), [2, 2])
    ]
    if len(triples) > 1:
        principal = set(ranked[:math.ceil(0.7 * len(ranked))])
        triples = [triple for triple in triples if set(triple) <= principal]
    gong = int(triples[0][0]) if triples else int(profile.argmax())
    return gong, np.roll(profile, -gong)


def evaluate_modes(table, model):
    raw_predictions = model["mode_predictions"]
    counts = pd.crosstab(pd.Series(raw_predictions), table["diao"])
    best_labels = max(
        permutations(counts.columns),
        key=lambda labels: sum(
            counts.loc[mode, label] for mode, label in zip(counts.index, labels)
        ),
    )
    label_map = dict(zip(counts.index, best_labels))
    predicted_labels = pd.Series(raw_predictions).map(label_map)
    precision, recall, f1, support = precision_recall_fscore_support(
        table["diao"], predicted_labels, labels=DIAO, zero_division=0
    )
    metrics = pd.DataFrame({
        "diao": DIAO, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    metrics.loc[len(metrics)] = [
        "macro", precision.mean(), recall.mean(), f1.mean(), support.sum()
    ]

    prevalence = [
        round(count / model["mode_counts"].sum() * 100, 2)
        for count in model["mode_counts"]
    ]
    mode_order = pd.Series(prevalence).sort_values(ascending=False).index
    mode_map = {raw_mode: mode_id for mode_id, raw_mode in enumerate(mode_order)}
    predictions = table[["sample_id", "name"]].copy()
    predictions["root_pc"] = model["root_predictions"]
    predictions["mode_id"] = pd.Series(raw_predictions).map(mode_map)

    confusion = pd.crosstab(table["diao"], predictions["mode_id"]).reindex(
        index=DIAO, columns=range(3), fill_value=0
    )
    confusion = confusion.rename_axis(index="diao", columns="mode_id").stack()
    confusion = confusion.rename("count").reset_index()

    mode_rows = []
    profile_rows = []
    for mode_id, raw_mode in enumerate(mode_order):
        mode_counts = model["mode_pitch_counts"][raw_mode]
        profile = mode_counts / mode_counts.sum()
        gong, aligned = align_mode_profile(profile)
        mode_rows.append({
            "mode_id": mode_id,
            "matched_diao": label_map[raw_mode],
            "prevalence_percent": prevalence[raw_mode],
            "reported_percent": float(f"{prevalence[raw_mode]:.1f}"),
            "predicted_pieces": int((predictions["mode_id"] == mode_id).sum()),
            "alignment_pc": gong,
        })
        for pitch, value in enumerate(aligned):
            profile_rows.append({
                "mode_id": mode_id, "pitch_class": pitch,
                "relative_duration": value,
            })
    return {
        "predictions": predictions, "metrics": metrics, "confusion_counts": confusion,
        "modes": pd.DataFrame(mode_rows), "mode_profiles": pd.DataFrame(profile_rows),
    }

## 6. Fit the three conditions

Each condition starts for its three-mode fit. The silhouette sweep
starts a new stream, fits two modes, uses the three-mode fit, and
continues through 4–21 modes. Each fit uses 1,000 sweeps and 100 warmup sweeps.

In [ ]:
results = {}
for condition, table in datasets.items():
    profiles = table[PITCH_COLUMNS].to_numpy()
    distances = mode_distance_matrix(profiles)
    model = fit_mode_model(
        profiles, n_modes=3, seed=SEED, epochs=EPOCHS, warmup=WARMUP
    )
    results[condition] = evaluate_modes(table, model)
    print(f"{condition}: three-mode fit complete", flush=True)

    silhouette_rows = []
    set_seed(SEED)
    for n_modes in range(2, 22):
        fitted = model if n_modes == 3 else fit_mode_model(
            profiles, n_modes=n_modes, seed=None, epochs=EPOCHS, warmup=WARMUP
        )
        score = silhouette_score(
            distances, fitted["mode_predictions"], metric="precomputed"
        )
        silhouette_rows.append({"k": n_modes, "silhouette": score})
        print(f"{condition}: K={n_modes:2d}, silhouette={score:.6f}", flush=True)
    results[condition]["silhouettes"] = pd.DataFrame(silhouette_rows)

## 7. Results

The tables below show classification metrics, confusion counts, inferred modes, and silhouette scores. The `combined` dictionary also contains the individual root/mode predictions (`predictions`) and inferred pitch-class profiles (`mode_profiles`).

In [ ]:
combined = {}
for table_name in [
    "predictions", "metrics", "confusion_counts", "modes",
    "mode_profiles", "silhouettes",
]:
    combined[table_name] = pd.concat([
        table[table_name].assign(condition=condition)
        for condition, table in results.items()
    ], ignore_index=True)
    columns = ["condition"] + [
        column for column in combined[table_name].columns if column != "condition"
    ]
    combined[table_name] = combined[table_name][columns]

silhouette_summary = (
    combined["silhouettes"].groupby("condition", sort=False)["silhouette"]
    .agg(mean_silhouette="mean", max_silhouette="max").reset_index()
)
reported_prevalence = combined["modes"][["condition", "mode_id", "reported_percent"]]

display(combined["metrics"].round(3))
display(combined["confusion_counts"].pivot(
    index=["condition", "diao"], columns="mode_id", values="count"
))
display(combined["modes"].round(3))
display(silhouette_summary.round(6))

display(combined["silhouettes"].pivot(
    index="k", columns="condition", values="silhouette"
).round(6))

## References

Meng, S., Moss, F. C., & Rohrmeier, M. A. (2026) *Revisiting Tong Yun San Gong Theory:
Corpus Evidence for the Distinctions among Three Chinese Heptatonic Diao*.

Harasim, D., Moss, F. C., Ramirez, M., & Rohrmeier, M. (2021).
Exploring the foundations of tonality: Statistical cognitive modeling of modes
in the history of Western classical music. *Humanities and Social Sciences
Communications, 8*, 5. https://doi.org/10.1057/s41599-020-00678-6.
[Model source](https://github.com/DCMLab/HistoryModes_DataCode).

Schaffrath, H. *Essen Folksong Collection*.
[Chinese collection in **kern format](https://kern.humdrum.org/cgi-bin/ksbrowse?l=/essen/asia/china).